In [2]:
import pandas as pd

df = pd.read_csv("googleplaystore.csv")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'googleplaystore.csv'

In [ ]:
df.shape

(10841, 13)

In [ ]:
df.columns

Index(['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type',
       'Price', 'Content Rating', 'Genres', 'Last Updated', 'Current Ver',
       'Android Ver'],
      dtype='str')

In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10841 entries, 0 to 10840
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             10841 non-null  str    
 1   Category        10841 non-null  str    
 2   Rating          9367 non-null   float64
 3   Reviews         10841 non-null  str    
 4   Size            10841 non-null  str    
 5   Installs        10841 non-null  str    
 6   Type            10840 non-null  str    
 7   Price           10841 non-null  str    
 8   Content Rating  10840 non-null  str    
 9   Genres          10841 non-null  str    
 10  Last Updated    10841 non-null  str    
 11  Current Ver     10833 non-null  str    
 12  Android Ver     10838 non-null  str    
dtypes: float64(1), str(12)
memory usage: 1.1 MB


In [ ]:
df.describe()

,Rating
count,9367.000000
mean,4.193338
std,0.537431
min,1.000000
25%,4.000000
50%,4.300000
75%,4.500000
max,19.000000


In [ ]:
df[df["Rating"] > 5]

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
10472,Life Made WI-Fi Touchscreen Photo Frame,1.9,19.0,3.0M,"1,000+",Free,0,Everyone,NaN,11-Feb-18,1.0.19,4.0 and up,NaN


Notice that the columns are misaligned. Either remove or shift right depends on usage.
Should be removed since isolated case.

In [ ]:
df = df.drop(index=10472)
df["Rating"].describe()

count    9366.000000
mean        4.191757
std         0.515219
min         1.000000
25%         4.000000
50%         4.300000
75%         4.500000
max         5.000000
Name: Rating, dtype: float64

In [ ]:
df["Rating"].isna().sum()

np.int64(1474)

1474/10841=13.5%. This is a significant number. Drop since analyzing ratings.

In [ ]:
df = df.dropna(subset=["Rating"])

Since we are analyzing which app category is rated highest by users, we can use groupby method


In [ ]:
df.groupby("Category")["Rating"].mean().sort_values(ascending=False).head(10)

Category
EVENTS                 4.435556
EDUCATION              4.389032
ART_AND_DESIGN         4.358065
BOOKS_AND_REFERENCE    4.346067
PERSONALIZATION        4.335987
PARENTING              4.300000
GAME                   4.286326
BEAUTY                 4.278571
HEALTH_AND_FITNESS     4.277104
SHOPPING               4.259664
Name: Rating, dtype: float64

This method only shows highest rating categories but doesn't consider about the volume of each category, which can lead to misinterpreting. We can use 'agg()' instead of 'mean()' to interpret more efficiency.

In [ ]:
df.groupby("Category")["Rating"].agg(["mean", "count"]).sort_values("mean", ascending=False).head(10)

,mean,count
Category,,
EVENTS,4.435556,45
EDUCATION,4.389032,155
ART_AND_DESIGN,4.358065,62
BOOKS_AND_REFERENCE,4.346067,178
PERSONALIZATION,4.335987,314
PARENTING,4.300000,50
GAME,4.286326,1097
BEAUTY,4.278571,42
HEALTH_AND_FITNESS,4.277104,297


Filter catogories that have more than 100 apps for better analysis

In [ ]:
df.groupby("Category")["Rating"].agg(["mean", "count"]).query("count >= 100").sort_values("mean", ascending=False).head(10)

,mean,count
Category,,
EDUCATION,4.389032,155
BOOKS_AND_REFERENCE,4.346067,178
PERSONALIZATION,4.335987,314
GAME,4.286326,1097
HEALTH_AND_FITNESS,4.277104,297
SHOPPING,4.259664,238
SOCIAL,4.255598,259
SPORTS,4.223511,319
PRODUCTIVITY,4.211396,351


Depends business goal, if we're going with the scale strategy, we focus in the GAME category, where the volume is high but also mean: oversaturated market, hard to compete, lower profit margins. Or we can go with the high-satisfaction strategy, which is EDUCATION: high satisfaction, less saturation, niche growth opportunity.

Now we check market size


In [ ]:
df["Installs"].dtype

<StringDtype(storage='python', na_value=nan)>

Since it's string type, we need to clean the data to integer to calculate the market size. 

In [ ]:
df["Installs"] = df["Installs"].astype(str).str.replace("+", "", regex=False).str.replace(",", "", regex=False)
df["Installs"] = pd.to_numeric(df["Installs"], errors="coerce")
df["Installs"].dtype

dtype('float64')

In [ ]:
df.groupby("Category")["Installs"].sum().sort_values(ascending=False).head(10)

Category
GAME                  3.508602e+10
COMMUNICATION         3.264728e+10
PRODUCTIVITY          1.417609e+10
SOCIAL                1.406987e+10
TOOLS                 1.145277e+10
FAMILY                1.025826e+10
PHOTOGRAPHY           1.008825e+10
NEWS_AND_MAGAZINES    7.496318e+09
TRAVEL_AND_LOCAL      6.868887e+09
VIDEO_PLAYERS         6.222003e+09
Name: Installs, dtype: float64

We see that game category dominates so this confirms that the category is in huge demand with massive user base and high engagement market.